In [2]:
!pip install -q langchain langchain-groq gspread

In [3]:
import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq connected successfully!")

Groq connected successfully!


In [5]:
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

ws.append_row([
    "name",
    "item",
    "qty",
    "date",
    "eggless",
    "total"
])

print("Sheet created successfully!")
print(sh.url)

Sheet created successfully!
https://docs.google.com/spreadsheets/d/1EqexKnVUWBzQTLKn1zyL1RtM5Hrv1zEcdvf86Rdss2Y


In [6]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}


class Order(BaseModel):
    """Stores a validated bakery order."""

    name: str
    item: Literal["plum cake", "chocolate cake", "cupcake"]
    qty: int = Field(ge=1)
    date: str
    eggless: bool


@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a menu item.

    Args:
        item: Name of the bakery item.
        qty: Number of items required.
    """

    price = MENU.get(item.lower().strip())

    if price is None:
        return "Item not available. Choose from the menu."

    if qty < 1:
        return "Quantity must be at least 1."

    return f"Total price: Rs {price * qty}"


@tool
def is_open(day: str) -> str:
    """Check the bakery opening hours for a particular weekday.

    Args:
        day: Day of the week.
    """

    hours = HOURS.get(day.lower().strip())

    if hours:
        return f"Open from {hours} on {day}."

    return "Invalid day."


@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """Save a validated customer order into Google Sheets.

    Args:
        name: Customer name.
        item: Bakery item.
        qty: Quantity.
        date: Pickup date.
        eggless: Whether eggless cake is required.
    """

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Order rejected: {e}"

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return f"Order saved! Total: Rs {total}"


print("All three bakery tools are ready!")

All three bakery tools are ready!


In [7]:
print(get_price.invoke({
    "item": "plum cake",
    "qty": 2
}))

print(is_open.invoke({
    "day": "sunday"
}))

Total price: Rs 1000
Open from 10 am to 6 pm on sunday.


In [8]:
print(get_price.invoke({
    "item": "donut",
    "qty": 2
}))

Item not available. Choose from the menu.


In [9]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly order assistant for Sweet Crumbs Home Bakery.

FACTS:
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Always use get_price for prices.
Always use is_open for opening hours.
Never guess prices or timings.
Before saving an order, collect the customer's name,
item, quantity and pickup date.
Ask for missing details.
Only save complete and valid orders.
After saving, confirm the total.

FORMAT:
Answer in under 60 words.
Be friendly and polite.
Reply in English or Telugu according to the customer.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []

def chat(text):
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, please try again in a minute."


print("Sweet Crumbs agent is ready!")

Sweet Crumbs agent is ready!


In [10]:
print(chat(
    "Are you open on Sunday? "
    "How much are 2 chocolate cakes?"
))

Yes, we’re open on Sunday from 10 am to 6 pm. Two chocolate cakes cost Rs 1,300. Remember, we need 24 hours’ notice for orders. Would you like to place an order? If so, please share your name, pickup date, and whether you’d like an eggless version.


In [11]:
print(chat(
    "I'm Ravi. Please book 2 eggless chocolate cakes for 24 December."
))

Great! Your order for 2 eggless chocolate cakes on 24 December is confirmed. Total: Rs 1,300. We’ll see you on the pickup day. Let us know if anything changes. Have a wonderful day!


In [12]:
history = []
print(chat("I want to order 2 donuts."))

Sure thing! Could you please share your name and the pickup date you’d like? (We need 24‑hour notice.) Once I have that, I’ll calculate the total and confirm your order.


In [13]:
history = []
print(chat("My name is Sravani. I want 3 cupcakes."))

Hi Sravani! Great choice. Could you let me know the pickup date you’d like, and whether you’d prefer eggless cupcakes? Once I have those details, I’ll confirm the price and place your order.


In [14]:
history = []
print(chat("I'm Arjun. Book 0 cupcakes for Saturday."))

Hi Arjun! 👋 To place your order, could you let me know how many cupcakes you’d like (the 0 you mentioned seems like a typo) and if you’d prefer an egg‑less version? Your pickup date is Saturday. Remember we need 24‑hour notice, so we’ll confirm once we have the details.


In [15]:
for m in history:
    m.pretty_print()

================================ Human Message =================================

I'm Arjun. Book 0 cupcakes for Saturday.
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_1263ad19-3820-4342-a230-547cf73131a4)
 Call ID: fc_1263ad19-3820-4342-a230-547cf73131a4
  Args:
    day: Saturday
================================= Tool Message =================================
Name: is_open

Open from 9 am to 8 pm on Saturday.
================================== Ai Message ==================================

Hi Arjun! 👋 To place your order, could you let me know how many cupcakes you’d like (the 0 you mentioned seems like a typo) and if you’d prefer an egg‑less version? Your pickup date is Saturday. Remember we need 24‑hour notice, so we’ll confirm once we have the details.


In [16]:
PACKAGES = {
    "goa": 12000,
    "hyderabad": 8000,
    "kerala": 15000
}


def get_package_price(
    destination: str,
    travellers: int = 1
) -> int:
    """Calculate the total sample package price."""

    destination = destination.lower().strip()

    if destination not in PACKAGES:
        raise ValueError("Destination not available.")

    if travellers < 1:
        raise ValueError("Travellers must be at least 1.")

    return PACKAGES[destination] * travellers


def check_travel_details(
    destination: str,
    travel_date: str
) -> str:
    """Check whether the destination and travel date are provided."""

    if destination.lower().strip() not in PACKAGES:
        return "Destination not available."

    if not travel_date.strip():
        return "Please provide a travel date."

    return "Travel details received. Availability needs confirmation."


print(get_package_price("goa", 2))
print(check_travel_details("goa", "24 December"))

24000
Travel details received. Availability needs confirmation.


In [17]:
from langchain.tools import tool


@tool
def get_package_price(destination: str, travellers: int = 1) -> str:
    """Calculate the total sample package price for a destination.

    Args:
        destination: Travel destination.
        travellers: Number of travellers.
    """

    destination = destination.lower().strip()

    if destination not in PACKAGES:
        return "Destination not available."

    if travellers < 1:
        return "Number of travellers must be at least 1."

    total = PACKAGES[destination] * travellers

    return f"Total package price: Rs {total}"


@tool
def check_travel_details(destination: str, travel_date: str) -> str:
    """Check whether a destination and travel date are provided.

    Args:
        destination: Travel destination.
        travel_date: Intended travel date.
    """

    if destination.lower().strip() not in PACKAGES:
        return "Destination not available."

    if not travel_date.strip():
        return "Please provide a travel date."

    return "Details received. Actual availability must be confirmed."


print(get_package_price.invoke({
    "destination": "goa",
    "travellers": 2
}))

Total package price: Rs 24000


In [18]:
from pydantic import BaseModel, Field
from typing import Literal


class TravelBooking(BaseModel):
    """Stores a validated travel booking enquiry."""

    name: str

    destination: Literal[
        "goa",
        "hyderabad",
        "kerala"
    ]

    travellers: int = Field(ge=1)

    travel_date: str


print("TravelBooking model created!")

TravelBooking model created!


In [19]:
booking_extractor = llm.with_structured_output(TravelBooking)

messages = [
    "Hi, I'm Ravi. I want to visit Goa with 2 people on 24 December.",

    "My name is Sravani. Plan a Kerala trip for 4 travellers on 30 December.",

    "Arjun here, I need a Hyderabad trip for 3 people on 5 January.",

    "Hello, I'm Priya. I want to visit Goa alone on 10 January.",

    "This is Kiran. I want a Kerala trip for 2 people on 15 January."
]

results = []

for message in messages:
    try:
        booking = booking_extractor.invoke(message)
        results.append(booking)

        print("Original:", message)
        print("Structured output:", booking)
        print("----------------------")

    except Exception as e:
        print("Extraction failed:", e)

Original: Hi, I'm Ravi. I want to visit Goa with 2 people on 24 December.
Structured output: name='Ravi' destination='goa' travellers=2 travel_date='24 December'
----------------------
Original: My name is Sravani. Plan a Kerala trip for 4 travellers on 30 December.
Structured output: name='Sravani' destination='kerala' travellers=4 travel_date='30 December'
----------------------
Original: Arjun here, I need a Hyderabad trip for 3 people on 5 January.
Structured output: name='Arjun' destination='hyderabad' travellers=3 travel_date='5 January'
----------------------
Original: Hello, I'm Priya. I want to visit Goa alone on 10 January.
Structured output: name='Priya' destination='goa' travellers=1 travel_date='10 January'
----------------------
Original: This is Kiran. I want a Kerala trip for 2 people on 15 January.
Structured output: name='Kiran' destination='kerala' travellers=2 travel_date='15 January'
----------------------


In [20]:
travel_sh = gc.create("Travel Booking Enquiries")

travel_ws = travel_sh.sheet1

travel_ws.append_row([
    "name",
    "destination",
    "travellers",
    "travel_date"
])

print("Travel sheet created!")
print(travel_sh.url)

Travel sheet created!
https://docs.google.com/spreadsheets/d/1DyynO8thibLt0QL2KOLwEJAaaryZWguXv9jsrZFNYrs


In [21]:
@tool
def log_travel_booking(
    name: str,
    destination: str,
    travellers: int,
    travel_date: str
) -> str:
    """Save a validated travel booking enquiry to Google Sheets.

    Args:
        name: Customer name.
        destination: Travel destination.
        travellers: Number of travellers.
        travel_date: Intended travel date.
    """

    try:
        booking = TravelBooking(
            name=name,
            destination=destination.lower().strip(),
            travellers=travellers,
            travel_date=travel_date
        )

    except Exception as e:
        return f"Booking rejected: {e}"

    travel_ws.append_row([
        booking.name,
        booking.destination,
        booking.travellers,
        booking.travel_date
    ])

    return "Travel enquiry saved successfully!"

In [22]:
print(log_travel_booking.invoke({
    "name": "Ravi",
    "destination": "goa",
    "travellers": 2,
    "travel_date": "24 December"
}))

Travel enquiry saved successfully!


In [23]:
travel_agent = create_agent(
    model=llm,
    tools=[
        get_package_price,
        check_travel_details,
        log_travel_booking
    ],
    system_prompt="""
    ROLE:
    You are a friendly travel booking assistant.

    FACTS:
    Sample destinations are Goa, Hyderabad and Kerala.

    RULES:
    Use get_package_price for package calculations.
    Use check_travel_details to check the supplied details.
    Ask for missing information.
    Never claim actual travel availability is confirmed.
    Save a booking enquiry only after collecting
    the customer's name, destination, traveller count and date.
    Use log_travel_booking to save the enquiry.

    FORMAT:
    Be friendly, clear and polite.
    """
)

print("Travel agent created!")

Travel agent created!


In [24]:
travel_result = travel_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "I'm Ravi. I want to visit Goa with 2 people "
                "on 24 December. Please check the price "
                "and save my enquiry."
            )
        }
    ]
})

for m in travel_result["messages"]:
    m.pretty_print()

================================ Human Message =================================

I'm Ravi. I want to visit Goa with 2 people on 24 December. Please check the price and save my enquiry.
================================== Ai Message ==================================
Tool Calls:
  get_package_price (fc_e85bfb38-f48a-43e8-9816-20fd45037ad9)
 Call ID: fc_e85bfb38-f48a-43e8-9816-20fd45037ad9
  Args:
    destination: Goa
    travellers: 2
================================= Tool Message =================================
Name: get_package_price

Total package price: Rs 24000
================================== Ai Message ==================================
Tool Calls:
  check_travel_details (fc_30a382e0-c978-4b29-a37a-84d0d58dfed0)
 Call ID: fc_30a382e0-c978-4b29-a37a-84d0d58dfed0
  Args:
    destination: Goa
    travel_date: 24 December
================================= Tool Message =================================
Name: check_travel_details

Details received. Actual availability must be confi

In [27]:
@tool
def tool1(destination: str, travellers: int = 1) -> str:
    """does stuff"""

    if destination.lower().strip() not in PACKAGES:
        return "Destination not available."

    if travellers < 1:
        return "Invalid number of travellers."

    return str(PACKAGES[destination.lower().strip()] * travellers)

In [26]:
bad_agent = create_agent(
    model=llm,
    tools=[tool1, check_travel_details],
    system_prompt="You are a travel assistant. Use tools when needed."
)

In [28]:
bad_result = bad_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the package price for 2 people visiting Goa?"
        }
    ]
})

for m in bad_result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is the package price for 2 people visiting Goa?
================================== Ai Message ==================================
Tool Calls:
  tool1 (fc_302cf9f8-0180-49c9-aa46-e6b72911def9)
 Call ID: fc_302cf9f8-0180-49c9-aa46-e6b72911def9
  Args:
    destination: Goa
    travellers: 2
================================= Tool Message =================================
Name: tool1

24000
================================== Ai Message ==================================

The package price for 2 people visiting Goa is **₹24,000**.


In [29]:
good_agent = create_agent(
    model=llm,
    tools=[
        get_package_price,
        check_travel_details,
        log_travel_booking
    ],
    system_prompt="You are a helpful travel assistant. Use the correct tool for each task."
)

print("Proper tool names restored!")

Proper tool names restored!


In [30]:
tool_call_count = 0
model_call_count = 0

for message in travel_result["messages"]:

    if message.type == "ai":
        model_call_count += 1

        tool_call_count += len(
            getattr(message, "tool_calls", [])
        )

print("Model calls:", model_call_count)
print("Tool calls:", tool_call_count)

Model calls: 4
Tool calls: 3


In [31]:
for message in travel_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

I'm Ravi. I want to visit Goa with 2 people on 24 December. Please check the price and save my enquiry.
================================== Ai Message ==================================
Tool Calls:
  get_package_price (fc_e85bfb38-f48a-43e8-9816-20fd45037ad9)
 Call ID: fc_e85bfb38-f48a-43e8-9816-20fd45037ad9
  Args:
    destination: Goa
    travellers: 2
================================= Tool Message =================================
Name: get_package_price

Total package price: Rs 24000
================================== Ai Message ==================================
Tool Calls:
  check_travel_details (fc_30a382e0-c978-4b29-a37a-84d0d58dfed0)
 Call ID: fc_30a382e0-c978-4b29-a37a-84d0d58dfed0
  Args:
    destination: Goa
    travel_date: 24 December
================================= Tool Message =================================
Name: check_travel_details

Details received. Actual availability must be confi

In [32]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

## Rate Limits and HTTP 429

Rate limits restrict how many API requests can be made during a particular time period.

HTTP 429 means Too Many Requests. It occurs when the application sends more requests than the provider currently allows.

An agent may make multiple LLM calls for one customer question because it may need to request a tool, receive its result and generate a final response.

### Handling
- Use max_retries to retry certain temporary API failures.
- Use try/except to handle errors gracefully.
- Ask users to wait before retrying.
- Avoid sending unnecessary repeated requests.
- Use individual API keys when required instead of sharing one key among many users.

### Learning Outcome
Understood why agents can use more API requests than a simple chatbot and how retries and error handling help manage temporary failures.

In [33]:
%%writefile business_facts.txt
BUSINESS: Travel and Tour Booking Platform

1. The platform provides travel package information.
2. Customers can explore different destinations.
3. Customers can submit booking enquiries.
4. Package prices depend on the destination.
5. Travel dates are required for booking enquiries.
6. Customers must provide the number of travellers.
7. Booking enquiries require a customer name.
8. Goa is one of the sample destinations.
9. Kerala is one of the sample destinations.
10. Hyderabad is one of the sample destinations.
11. Package prices used in the demo are sample prices.
12. Actual travel availability must be confirmed.
13. Customers should verify their travel dates.
14. Customers should verify package details.
15. Booking enquiries are stored in Google Sheets.
16. Each enquiry is saved as a separate row.
17. Invalid traveller counts are rejected.
18. Unsupported destinations are rejected.
19. Pydantic validates booking information.
20. The assistant can ask customers for missing details.

Writing business_facts.txt


In [34]:
from google.colab import files

files.download("business_facts.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>